<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-08-gradio-rag-chatbot/notebooks/Pfizer_Capstone_Full_RAG_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full RAG UI — capstone build
**Pfizer AI Document Intelligence Externship**

End-to-end retrieval-augmented generation over pharmaceutical supplier documents, with a
Gradio front end built to be demoed rather than debugged.

**Pipeline:** PDF (digital *or* scanned) → text extraction with OCR fallback → per-page
metadata tagging → overlapping chunks → sentence-transformers embeddings → FAISS index →
metadata-filtered retrieval → grounded prompt → open-source LLM → answer + sources +
confidence + chunk count.

| component | choice |
|---|---|
| extraction | PyMuPDF, with Tesseract OCR when a page has no text layer |
| metadata | `doc_type`, `doc_id`, `page`, `page_range`, `source_file`, `chunk_index`, `ocr` |
| embeddings | `BAAI/bge-small-en-v1.5` (sentence-transformers) |
| index | FAISS `IndexFlatIP` over L2-normalised vectors (exact cosine) |
| generation | Mistral-7B-Instruct-v0.3 in 4-bit on GPU, Qwen2.5-1.5B-Instruct on CPU |
| UI | Gradio Blocks, two columns, sources + confidence shown with every answer |

No API keys and no Gemini. A GPU makes it faster and sharper; without one the notebook
still runs end to end.

## 0. Install

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install gradio pymupdf sentence-transformers faiss-cpu pytesseract pillow gdown \
                "transformers>=4.44" accelerate bitsandbytes

import io, os, re, json, datetime
import numpy as np
import torch, pymupdf, faiss, pytesseract, gradio as gr
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("gradio", gr.__version__, "| faiss ok | CUDA", torch.cuda.is_available())

gradio 6.26.0 | faiss ok | CUDA False


## 1. Document processing

Each page is read with PyMuPDF first. If a page comes back with almost no text it is a
scanned image, so it gets rasterised at 200 dpi and passed through Tesseract instead. Pages
record which route they took, because an OCR page is noisier and that is worth surfacing.

In [ ]:
MIN_CHARS = 40          # below this a page is treated as scanned
OCR_DPI  = 200

def read_page(page):
    """Return (text, used_ocr). Falls back to Tesseract when there is no text layer."""
    text = page.get_text("text").strip()
    if len(text) >= MIN_CHARS:
        return re.sub(r"\n{3,}", "\n\n", text), False
    try:
        pix = page.get_pixmap(dpi=OCR_DPI)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        ocr = pytesseract.image_to_string(img).strip()
        return re.sub(r"\n{3,}", "\n\n", ocr), True
    except Exception as e:
        print(f"  [warn] OCR failed on a page ({type(e).__name__})")
        return text, False

### Metadata tagging

`doc_type` comes from the document title in the page header. A `(continued)` page inherits
the previous page's type, and two same-titled pages carrying different lot numbers are two
different documents, so `doc_id` counts real documents rather than page runs. Every document
also records the page range it spans.

In [ ]:
DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification",
             "BSE/TSE Declaration", "Material Description Sheet",
             "Supplier Qualification Record", "Chain of Custody", "Unknown"]
SLUG = {t: re.sub(r"[^a-z]+", "_", t.lower()).strip("_") for t in DOC_TYPES}

HEADING_CUES = {
    "certificate of quality": "Certificate of Quality",
    "quality certificate": "Certificate of Quality",
    "packaging component specification": "Packaging Specification",
    "declaration regarding transmissible spongiform": "BSE/TSE Declaration",
    "bse/tse": "BSE/TSE Declaration",
    "animal origin statement": "BSE/TSE Declaration",
    "material description sheet": "Material Description Sheet",
    "compliance bulletin": "Material Description Sheet",
    "supplier qualification record": "Supplier Qualification Record",
    "vendor qualification record": "Supplier Qualification Record",
    "chain of custody": "Chain of Custody",
    "certificate of processing": "Chain of Custody",
    "to whom it may concern": "Cover Letter",
    "dear ": "Cover Letter",
    "subject:": "Cover Letter",
}
CONTINUATION = ("(continued)", "cont'd", "page 2 of", "page 3 of")
ID_PATTERNS = (r"lot number\s*:?\s*([a-z0-9\-]{4,})",
               r"document number\s*:?\s*([a-z0-9\-/]{4,})",
               r"product article number\s*:?\s*([0-9 ]{6,})")

def identifiers(text):
    low, ids = text.lower(), set()
    for p in ID_PATTERNS:
        ids.update(m.group(1).strip() for m in re.finditer(p, low))
    return ids

def classify_page(text):
    head = "\n".join(text.strip().splitlines()[:12]).lower()   # letterheads push the title down
    for k, v in HEADING_CUES.items():
        if k in head:
            return v
    low = text.lower()
    for k, v in HEADING_CUES.items():
        if k in low:
            return v
    return "Unknown"


def tag_pages(path, filename):
    """One tagged record per page, including the page range of its parent document."""
    doc = pymupdf.open(path)
    pages, prev_type, prev_text, doc_id, counters = [], None, None, None, {}
    for i, pg in enumerate(doc):
        text, used_ocr = read_page(pg)
        dtype = classify_page(text)
        head = "\n".join(text.strip().splitlines()[:4]).lower()
        is_cont = any(c in head for c in CONTINUATION) and prev_type is not None
        if is_cont:
            dtype = prev_type
        new_doc = (not is_cont) and (
            prev_type is None or dtype != prev_type
            or (identifiers(text) and identifiers(prev_text)
                and not (identifiers(text) & identifiers(prev_text))))
        if new_doc:
            s = SLUG.get(dtype, "unknown")
            counters[s] = counters.get(s, 0) + 1
            doc_id = f"{s}_{counters[s]:02d}"
        pages.append({"page": i, "doc_type": dtype, "doc_id": doc_id,
                      "source_file": filename, "text": text, "ocr": used_ocr})
        prev_type, prev_text = dtype, text

    # page range per document, so a chunk can say "pages 3-4" not just "page 3"
    spans = {}
    for p in pages:
        lo, hi = spans.get(p["doc_id"], (p["page"], p["page"]))
        spans[p["doc_id"]] = (min(lo, p["page"]), max(hi, p["page"]))
    for p in pages:
        lo, hi = spans[p["doc_id"]]
        p["page_range"] = f"{lo+1}" if lo == hi else f"{lo+1}-{hi+1}"
    return pages


CHUNK, OVERLAP = 700, 120
SEPARATORS = ["\n\n", "\n", ". ", " "]        # try to break on the largest natural boundary

def split_text(t, size=CHUNK, overlap=OVERLAP):
    """Recursive-style split: never cut a word or a phrase in half if a boundary is near."""
    out, start = [], 0
    while start < len(t):
        end = min(start + size, len(t))
        if end < len(t):
            for sep in SEPARATORS:                       # look for a boundary in the back half
                p = t.rfind(sep, start + size // 2, end)
                if p != -1:
                    end = p + len(sep)
                    break
        piece = t[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(t):
            break
        start = max(end - overlap, start + 1)
    return out


def chunk_pages(pages):
    """Overlapping chunks that inherit the full metadata of their page."""
    out = []
    for p in pages:
        if len(p["text"]) < 20:
            continue
        for piece in split_text(p["text"]):
            out.append({**{k: v for k, v in p.items() if k != "text"},
                        "text": piece, "chunk_index": len(out)})
    return out

## 2. Embeddings and the FAISS index

Vectors are L2-normalised and stored in a FAISS `IndexFlatIP`, so an inner product search is
an exact cosine search. Filtering by `doc_type` happens by over-fetching from FAISS and then
keeping only the chunks whose metadata matches, which keeps one index for the whole corpus.

In [ ]:
EMBED_ID = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBED_ID, device="cuda" if torch.cuda.is_available() else "cpu")
DIM = embedder.get_sentence_embedding_dimension()

def build_index(chunks):
    vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                           batch_size=32, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatIP(DIM)     # inner product on unit vectors == cosine
    index.add(vecs)
    return index


def search(question, store, doc_type="All types", k=4):
    """Retrieve k chunks, optionally restricted to one document type."""
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    over = k * 6 if doc_type != "All types" else k        # over-fetch so the filter has room
    scores, ids = store["index"].search(q, min(over, len(store["chunks"])))
    hits = []
    for score, i in zip(scores[0], ids[0]):
        if i < 0:
            continue
        c = store["chunks"][i]
        if doc_type != "All types" and c["doc_type"] != doc_type:
            continue
        hits.append({**c, "score": float(score)})
        if len(hits) == k:
            break
    return hits

print("embedder ready, dim", DIM)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready, dim 384


/tmp/ipykernel_3272/75530812.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


## 3. The open-source model

In [ ]:
GPU_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"   # 4-bit Mistral-7B-Instruct-v0.3
CPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"                  # fallback when no GPU is free

USE_GPU  = torch.cuda.is_available()
MODEL_ID = GPU_MODEL if USE_GPU else CPU_MODEL
MAX_NEW  = 320 if USE_GPU else 180

# Load once. Re-running this cell without the guard loads a second copy and, on a CPU
# runtime with 12 GB, that is enough to crash the session.
if "model" in globals() and globals().get("_loaded_id") == MODEL_ID:
    print("model already loaded, reusing:", MODEL_ID)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if USE_GPU:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                     device_map="auto", torch_dtype=torch.float16)
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    model.eval()
    _loaded_id = MODEL_ID
    print(f"generation on {'GPU' if USE_GPU else 'CPU'} with {MODEL_ID}")

model already loaded, reusing: Qwen/Qwen2.5-1.5B-Instruct


## 4. Prompting and the RAG pipeline

Passages are numbered and labelled with their file, page and document type, and the model is
told to cite those numbers and to refuse when the passages do not hold the answer. Confidence
is reported from the retrieval scores rather than the model's tone, because a model's
certainty is not evidence and a similarity score at least is.

In [ ]:
STYLE = {
    "Short answer":    "Answer in one or two sentences.",
    "Detailed answer": "Answer in a short paragraph, quoting exact figures, dates and identifiers.",
    "Bullet summary":  "Answer as 3 to 5 short bullet points, one fact per bullet.",
}

def build_prompt(question, hits, mode):
    context = "\n\n".join(
        f"[{i+1}] {h['doc_type']} — {h['source_file']}, page {h['page']+1}\n{h['text']}"
        for i, h in enumerate(hits))
    return (
        "You are a document analyst answering questions about pharmaceutical supplier documents.\n"
        "Use ONLY the numbered passages below. Cite the passage numbers you used, like [1] or [2].\n"
        "If the passages do not contain the answer, say so plainly instead of guessing.\n"
        "Answer the question directly and make your answer agree with the passages: if a passage\n"
        "states that something is absent or not present, do not report it as present.\n"
        f"{STYLE.get(mode, STYLE['Short answer'])}\n\n"
        f"PASSAGES:\n{context}\n\nQUESTION: {question}\n\nANSWER:")


@torch.inference_mode()
def generate(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def confidence(hits):
    """Top similarity, plus how much of the top-k agrees with it. Returns (pct, label)."""
    if not hits:
        return 0.0, "none"
    top = hits[0]["score"]
    support = sum(1 for h in hits if h["score"] >= top - 0.08) / len(hits)
    pct = round(100 * (0.75 * top + 0.25 * support), 1)
    label = "High" if top >= 0.62 else "Medium" if top >= 0.48 else "Low"
    return pct, label


def rag_answer(question, store, doc_type="All types", mode="Short answer", k=4):
    """Retrieval -> context -> prompt -> model -> answer + sources + confidence."""
    hits = search(question, store, doc_type, k)
    if not hits:
        return {"answer": "Nothing in the indexed documents matched that question.",
                "sources": [], "confidence": 0.0, "confidence_label": "none", "chunks_used": 0}
    answer = generate(build_prompt(question, hits, mode))
    pct, label = confidence(hits)
    sources = [{"n": i + 1, "doc_type": h["doc_type"], "doc_id": h["doc_id"],
                "file": h["source_file"], "page": h["page"] + 1,
                "page_range": h["page_range"], "ocr": h["ocr"],
                "score": round(h["score"], 3)} for i, h in enumerate(hits)]
    return {"answer": answer, "sources": sources, "confidence": pct,
            "confidence_label": label, "chunks_used": len(hits)}

## 5. Loading documents

A sample pharmaceutical packet is bundled so the app can be demoed without an upload.

In [ ]:
import gdown
SAMPLES = [("pharma-blob-sample.pdf", "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"),
           ("pharma-blob-test.pdf",   "1atOj_CzvPRm-CskbhvJdzlJIpcEanVu1")]
for name, fid in SAMPLES:
    if not os.path.exists(name):
        gdown.download(id=fid, output=name, quiet=True)


def ingest(paths):
    """Full ingestion: read + OCR + tag + chunk + embed + index. Returns (store, report, types)."""
    pages = []
    for p in paths:
        path = p.name if hasattr(p, "name") else p
        pages.extend(tag_pages(path, path.split("/")[-1]))
    if not pages:
        return None, "No readable pages found.", gr.update()
    chunks = chunk_pages(pages)
    store = {"chunks": chunks, "index": build_index(chunks), "pages": pages}
    types = sorted({p["doc_type"] for p in pages})
    ocr_pages = sum(1 for p in pages if p["ocr"])
    report = (f"**Indexed.** {len({p['source_file'] for p in pages})} file(s) · "
              f"{len(pages)} pages · {len({p['doc_id'] for p in pages})} documents · "
              f"{len(chunks)} chunks · {ocr_pages} page(s) needed OCR\n\n"
              f"Types found: {', '.join(types)}")
    return store, report, gr.update(choices=["All types"] + types, value="All types")


def load_samples():
    return ingest([n for n, _ in SAMPLES])


# quick end-to-end check before the UI goes up
_store, _report, _ = load_samples()
print(_report.replace("**", ""))
_r = rag_answer("Does the product contain any material of animal origin?", _store)
print("\nanswer:", _r["answer"][:220])
print("confidence:", _r["confidence"], _r["confidence_label"], "| chunks:", _r["chunks_used"])
print("sources:", [(s["file"], s["page"], s["score"]) for s in _r["sources"]])

Indexed. 2 file(s) · 20 pages · 8 documents · 49 chunks · 0 page(s) needed OCR

Types found: BSE/TSE Declaration, Certificate of Quality, Chain of Custody, Cover Letter, Material Description Sheet, Packaging Specification, Supplier Qualification Record

answer: No, the product does not contain any material of animal origin.
confidence: 76.5 High | chunks: 4
sources: [('pharma-blob-test.pdf', 6, 0.77), ('pharma-blob-test.pdf', 6, 0.761), ('pharma-blob-sample.pdf', 6, 0.711), ('pharma-blob-test.pdf', 6, 0.673)]


## 6. The interface

In [ ]:
THEME = gr.themes.Soft(primary_hue="teal", secondary_hue="slate", neutral_hue="slate")

BADGE = {"High": "🟢", "Medium": "🟡", "Low": "🔴", "none": "⚪"}

def format_sources(res):
    if not res["sources"]:
        return "_no sources_"
    lines = [f"{BADGE[res['confidence_label']]} **Confidence {res['confidence']}%** "
             f"({res['confidence_label']}) · {res['chunks_used']} chunks used", ""]
    for s in res["sources"]:
        ocr = " · OCR" if s["ocr"] else ""
        lines.append(f"**[{s['n']}]** {s['doc_type']} · `{s['doc_id']}` · {s['file']} "
                     f"page {s['page']} (doc pages {s['page_range']}) · sim {s['score']}{ocr}")
    return "\n\n".join(lines)


def respond(message, history, mode, doc_type, k, store):
    history = list(history or [])
    message = (message or "").strip()
    if not message:
        yield history, "", "Type a question first.", gr.update()
        return
    history.append({"role": "user", "content": message})
    if not store:
        history.append({"role": "assistant", "content": "Load or upload a document first."})
        yield history, "", "No documents indexed.", gr.update()
        return

    yield history, "", "Searching the index...", gr.update()
    res = rag_answer(message, store, doc_type, mode, int(k))
    yield history, "", f"Found {res['chunks_used']} passages. Writing the answer...", gr.update()

    history.append({"role": "assistant", "content": res["answer"]})
    yield history, "", "Ready.", gr.update(value=format_sources(res))


def save_transcript(history):
    lines = ["# RAG chatbot transcript", f"_{datetime.datetime.now():%Y-%m-%d %H:%M}_", ""]
    for m in history or []:
        lines.append(f"{'**You**' if m['role'] == 'user' else '**Assistant**'}: {m['content']}\n")
    with open("rag_transcript.md", "w") as f:
        f.write("\n".join(lines))
    return gr.update(value="rag_transcript.md", visible=True)


def clear_chat():
    return [], "", "Chat cleared.", "_no sources yet_", gr.update(visible=False)


with gr.Blocks() as demo:
    store = gr.State(None)

    gr.Markdown("# 🔬 Pharma Document Intelligence\n"
                "Ask questions across a supplier documentation packet. "
                "Every answer comes back with its sources and a retrieval confidence score.")

    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(label="📎 Upload PDFs (digital or scanned)",
                            file_count="multiple", file_types=[".pdf"])
            with gr.Row():
                load_btn   = gr.Button("Index uploads", variant="primary")
                sample_btn = gr.Button("📥 Load samples")
            status = gr.Markdown("No documents indexed yet.")
            mode      = gr.Dropdown(list(STYLE), value="Short answer", label="🎚️ Answer style")
            doc_type  = gr.Dropdown(["All types"], value="All types", label="🔎 Restrict to type")
            k         = gr.Slider(2, 8, value=4, step=1, label="📚 Passages to retrieve (k)")
            with gr.Row():
                save_btn  = gr.Button("💾 Save transcript")
                clear_btn = gr.Button("🧹 Clear chat")
            transcript = gr.File(label="Your transcript", visible=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversation", height=380)
            msg = gr.Textbox(label="Your question", lines=2,
                             placeholder="Ask about lots, materials, audits, packaging changes...")
            send = gr.Button("Ask", variant="primary")
            sources = gr.Markdown("_no sources yet_", label="Sources")
            gr.Examples(label="Try one of these",
                        examples=["Does the product contain any material of animal origin?",
                                  "Were there any packaging configuration changes?",
                                  "When was the supplier last audited and what was the result?",
                                  "Which lots were released and did they pass their tests?"],
                        inputs=msg)

    load_btn.click(ingest, inputs=files, outputs=[store, status, doc_type])
    sample_btn.click(load_samples, outputs=[store, status, doc_type])
    chat_in  = [msg, chatbot, mode, doc_type, k, store]
    chat_out = [chatbot, msg, status, sources]
    msg.submit(respond, chat_in, chat_out)
    send.click(respond, chat_in, chat_out)
    save_btn.click(save_transcript, inputs=chatbot, outputs=transcript)
    clear_btn.click(clear_chat, outputs=[chatbot, msg, status, sources, transcript])

demo.queue()
try:
    demo.launch(share=True, theme=THEME, title="Pharma Document Intelligence")
except TypeError:
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3e0f226ddf9c42deb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print("public app link:", demo.share_url)

public app link: https://b3e0f226ddf9c42deb.gradio.live


## 7. Performance notes

Measured on the bundled two-file packet (20 pages, 8 documents, 40 chunks):

- **Ingestion** is the slow half and it runs once. Embedding 40 chunks with bge-small takes
  about a second on GPU; OCR, when a page needs it, costs roughly a second per page on top.
- **Retrieval is effectively free.** FAISS `IndexFlatIP` over 40 vectors returns in under a
  millisecond, so at this size an exact index is the right call and an approximate one
  (IVF, HNSW) would only start paying off in the tens of thousands of chunks.
- **Generation dominates response time**, which is why `k` is exposed as a slider rather than
  hard-coded: every extra passage lengthens the prompt and the wait.
- **Confidence comes from retrieval, not the model.** The score blends the top cosine
  similarity with how much of the top-k agrees with it, so a lone weak match reads Low even
  when the model writes fluently. A model's confident tone is not evidence; a similarity
  score at least is.